In [1]:
%pip install -q "transformers==4.57.6" "tokenizers==0.22.1" librosa soundfile speechbrain
#Installs the Python libraries needed to load, preprocess, and analyze audio using the pretrained deepfake detection model.

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 912.7/912.7 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 31.5 MB/s eta 0:00:00


In [2]:
# Uploads the two voice recordings used to test both pretrained models.
from google.colab import files

uploaded = files.upload()

print("Uploaded files:", list(uploaded.keys()))

Saving Aiden1.wav to Aiden1.wav
Saving Aiden2.wav to Aiden2.wav
Uploaded files: ['Aiden1.wav', 'Aiden2.wav']


In [3]:
# Converts both recordings to 16 kHz mono WAV audio to ensure compatibility with the models.
import librosa
import soundfile as sf
import os

audio_files = ["Aiden1.wav", "Aiden2.wav"]

for filename in audio_files:
    if not os.path.exists(filename):
        raise FileNotFoundError(f"Missing {filename}. Upload it in Cell 2.")

    audio, sr = librosa.load(filename, sr=16000, mono=True)

    sf.write(filename, audio, 16000)

    print(f"{filename}: {len(audio)/sr:.2f} seconds, 16000 Hz, mono")

print("Audio preprocessing complete.")

Aiden1.wav: 17.72 seconds, 16000 Hz, mono
Aiden2.wav: 16.42 seconds, 16000 Hz, mono
Audio preprocessing complete.


In [4]:
# Uses the pretrained SpeechBrain ECAPA-TDNN model to determine whether both recordings belong to the same speaker.
from speechbrain.inference.speaker import SpeakerRecognition

verification = SpeakerRecognition.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="pretrained_models/spkrec-ecapa-voxceleb"
)

score, prediction = verification.verify_files(
    "Aiden1.wav",
    "Aiden2.wav"
)

print("\n===== SPEAKER VERIFICATION =====")
print("Speaker 1: Aiden")
print("Speaker 2: Aiden")
print("Similarity score:", float(score))
print("Prediction:", int(prediction))

if int(prediction) == 1:
    print("Result: SAME SPEAKER")
else:
    print("Result: DIFFERENT SPEAKERS")

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


hyperparams.yaml: 0.00B [00:00, ?B/s]

INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


embedding_model.ckpt:   0%|          | 0.00/83.3M [00:00<?, ?B/s]

INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


mean_var_norm_emb.ckpt:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

... (2 outputs left out)

INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


label_encoder.txt: 0.00B [00:00, ?B/s]

INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder



===== SPEAKER VERIFICATION =====
Speaker 1: Aiden
Speaker 2: Aiden
Similarity score: 0.837629497051239
Prediction: 1
Result: SAME SPEAKER


In [5]:
# Uses a pretrained Wav2Vec2 model to predict whether each recording contains human or AI-generated speech.
import librosa
from transformers import pipeline

MODEL_ID = "Hemgg/Deepfake-audio-detection"

detector = pipeline(
    "audio-classification",
    model=MODEL_ID,
    device=-1
)

print("Model labels:", detector.model.config.id2label)

for filename in ["Aiden1.wav", "Aiden2.wav"]:

    audio, sr = librosa.load(
        filename,
        sr=16000,
        mono=True
    )

    results = detector(
        {"array": audio, "sampling_rate": sr},
        top_k=None
    )

    print(f"\n===== {filename} =====")
    print("Duration:", round(len(audio) / sr, 2), "seconds")

    for result in results:
        print(result["label"], f'{result["score"]:.2%}')

print("\nDeepfake detection inference complete.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Device set to use cpu


Model labels: {0: 'AIVoice', 1: 'HumanVoice'}

===== Aiden1.wav =====
Duration: 17.72 seconds
AIVoice 96.71%
HumanVoice 3.29%

===== Aiden2.wav =====
Duration: 16.42 seconds
AIVoice 92.81%
HumanVoice 7.19%

Deepfake detection inference complete.
